# Эксперимент 33: Discrete token models

**Статья:** Children's Speech Recognition through Discrete Token models (Распознавание детской речи через модели дискретных токенов) 2024

**Ссылка:** [https://arxiv.org/abs/2406.13431](https://arxiv.org/abs/2406.13431)

**Краткое описание модели:** Дискретизация акустических кадров: квантование MFCC в кодбук токенов (MiniBatchKMeans), unigram+bigram token-статистики и энтропийные признаки -> классификатор.

**Содержание статьи:** Реализован token-level pipeline: сигнал переводится в последовательность дискретных токенов, затем вычисляются вероятностные и переходные характеристики токенов для классификации дефектов речи.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
from sklearn.cluster import MiniBatchKMeans
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score, classification_report

exp_dir = Path().resolve()
sys.path.insert(0, str(exp_dir.parent))

from shared import config, data_utils
from shared.results_utils import save_result_csv

## 1. Загрузка данных и разбиение

In [ ]:
paths_train, paths_val, paths_test, y_train, y_val, y_test, letters_train, letters_val, letters_test = data_utils.get_splits()
print(f"Train: {len(paths_train)}, Val: {len(paths_val)}, Test: {len(paths_test)}")
n_letters = letters_train.shape[1]

Train: 1942, Val: 417, Test: 417


## 2. Подготовка признаков / представлений

In [3]:
N_TOKENS = 64

def extract_seq(path, n_mfcc=20, max_frames=320):
    y, sr = data_utils.load_audio(path, sr=config.TARGET_SR, max_sec=config.MAX_DURATION_SEC)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc, hop_length=config.HOP_LENGTH, n_fft=config.WIN_LENGTH).T
    if mfcc.shape[0] > max_frames:
        idx = np.linspace(0, mfcc.shape[0] - 1, max_frames).astype(int)
        mfcc = mfcc[idx]
    return mfcc.astype(np.float32)

seq_train = [extract_seq(p) for p in paths_train]
seq_val   = [extract_seq(p) for p in paths_val]
seq_test  = [extract_seq(p) for p in paths_test]

sample_frames = np.vstack(seq_train)
if sample_frames.shape[0] > 200000:
    idx = np.random.RandomState(config.RANDOM_STATE).choice(sample_frames.shape[0], 200000, replace=False)
    sample_frames = sample_frames[idx]
kmeans = MiniBatchKMeans(n_clusters=N_TOKENS, random_state=config.RANDOM_STATE, batch_size=4096, n_init=10)
kmeans.fit(sample_frames)


MiniBatchKMeans(batch_size=4096, n_clusters=64, n_init=10, random_state=42)

## 3. Обучение, оценка и сохранение результата

In [ ]:
def seq_to_token_features(seq):
    tokens = kmeans.predict(seq)
    uni = np.bincount(tokens, minlength=N_TOKENS).astype(np.float32)
    uni = uni / max(1.0, uni.sum())
    bi = np.zeros((N_TOKENS, N_TOKENS), dtype=np.float32)
    if len(tokens) > 1:
        for a, b in zip(tokens[:-1], tokens[1:]):
            bi[a, b] += 1.0
        bi /= max(1.0, bi.sum())
    entropy = -np.sum(uni * np.log(uni + 1e-8), dtype=np.float32)
    repeat_ratio = float(np.mean(tokens[1:] == tokens[:-1])) if len(tokens) > 1 else 0.0
    return np.concatenate([uni, bi.reshape(-1), np.array([entropy, repeat_ratio], dtype=np.float32)], axis=0)

X_train = np.stack([seq_to_token_features(s) for s in seq_train])
X_val   = np.stack([seq_to_token_features(s) for s in seq_val])
X_test  = np.stack([seq_to_token_features(s) for s in seq_test])

X_train = np.hstack([X_train, letters_train])
X_val   = np.hstack([X_val, letters_val])
X_test  = np.hstack([X_test, letters_test])

pipe = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(class_weight="balanced", max_iter=4000, solver="liblinear"))])
param_grid = {"clf__C": [0.1, 0.3, 1.0, 3.0, 10.0]}
grid = GridSearchCV(pipe, param_grid, cv=5, scoring="f1_macro", refit=True, n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)
clf = grid.best_estimator_

y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]
accuracy = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")
f1_bad = f1_score(y_test, y_pred, average="binary", pos_label=config.CLASS_BAD)
roc_auc = roc_auc_score(y_test, y_proba)
precision_bad = precision_score(y_test, y_pred, zero_division=0, pos_label=config.CLASS_BAD)
recall_bad = recall_score(y_test, y_pred, zero_division=0, pos_label=config.CLASS_BAD)

print(classification_report(y_test, y_pred, target_names=config.CLASS_NAMES))
display(pd.DataFrame([{"accuracy": accuracy, "f1_macro": f1_macro, "f1_bad": f1_bad, "roc_auc": roc_auc, "precision_bad": precision_bad, "recall_bad": recall_bad}]))

save_result_csv(
    exp_dir=exp_dir, 
    experiment_id="exp_33_discrete_token_proxy", 
    experiment_name="Discrete token models", 
    model="Token histogram+bigram LogReg", 
    accuracy=accuracy, 
    f1_macro=f1_macro, 
    f1_bad=f1_bad, 
    roc_auc=roc_auc, 
    precision_bad=precision_bad, 
    recall_bad=recall_bad, 
    notes=f"token_kmeans_k={N_TOKENS},grid={grid.best_params_}", 
    num_params=None, 
    train_time_sec=None
)

Fitting 5 folds for each of 5 candidates, totalling 25 fits
              precision    recall  f1-score   support

        good       0.78      0.71      0.74       282
         bad       0.49      0.59      0.54       135

    accuracy                           0.67       417
   macro avg       0.64      0.65      0.64       417
weighted avg       0.69      0.67      0.68       417



,accuracy,f1_macro,f1_bad,roc_auc,precision_bad,recall_bad
0,0.671463,0.6418,0.538721,0.683215,0.493827,0.592593


Результат сохранён в result.csv текущего эксперимента
